# Case Study: Gender Bias --- Localize, Fix, Re-Audit

**Goal:** Demonstrate the full responsible-AI loop: discover which circuit components encode gender bias, validate the finding, surgically mitigate it via selective fine-tuning, then re-audit to confirm the fix held *without* degrading general capability.

**Literature context:**
- [Identifying and Adapting Transformer-Components Responsible for Gender Bias](https://arxiv.org/abs/2310.12611) showed bias concentrates in specific attention heads.
- [Understanding and Mitigating Gender Bias in LLMs via Interpretable Neuron Editing](https://arxiv.org/abs/2501.14457) identified neuron circuits for bias and edited two key neurons to reduce it.
- [Dissecting Bias in LLMs: A Mechanistic Interpretability Perspective](https://arxiv.org/abs/2506.05166) confirmed bias is localized into certain edges and layers.

**What this notebook does (end-to-end, no placeholders):**
1. Measure baseline gender-bias signal via logit-diff on pronoun prediction
2. Discover the bias circuit with EAP-IG
3. Audit: run faithfulness evaluation to confirm the circuit is causal
4. Mitigate: use selective fine-tuning to identify and mask the bias-critical components
5. Re-audit: measure bias reduction + capability preservation

**Compute:** GPU recommended (Qwen2.5-1.5B-Instruct). A GPT-2 CPU fallback is included.

**Runtime:** ~20 min (GPU) / ~10 min (CPU, reduced examples)

---

### Why this matters

Bias audits that stop at "we found it" are incomplete. Regulators and internal review boards need the full loop: detection, mitigation, and *proof that the fix didn't break something else*. CircuitKit's evaluation framework provides that proof as a structured, reproducible artifact.

## 0. Setup

In [ ]:
import torch
import circuitkit as ck
from pathlib import Path

USE_GPU = torch.cuda.is_available()
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" if USE_GPU else "gpt2"
TASK = "gender_bias"
N_EXAMPLES = 128 if USE_GPU else 32
BATCH_SIZE = 8 if USE_GPU else 4
RESULTS_DIR = Path("./results/gender_bias")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model:    {MODEL_NAME}")
print(f"Device:   {'cuda' if USE_GPU else 'cpu'}")
print(f"Examples: {N_EXAMPLES}")

## 1. Baseline: Measure gender-bias signal

The `gender_bias` task uses minimal pairs where the model predicts a gendered pronoun. The logit-diff metric (`correct_gender - incorrect_gender`) captures how strongly the model favors one gender over another.

Backend: `circuitkit.tasks.builtins.gender_bias.GenderBiasTaskSpec` generates paired prompts and computes `_logit_diff` as the attribution metric.

In [ ]:
model = ck.load_model(MODEL_NAME, dtype="float32" if not USE_GPU else "bfloat16")
print(f"Loaded {MODEL_NAME} ({model.cfg.n_layers} layers, {model.cfg.n_heads} heads)")

In [ ]:
from circuitkit.tasks.registry import get_task
from circuitkit.tasks.bootstrap import _bootstrap_builtin_tasks
from circuitkit.backends.eap.eap_utils import tokenize_batch_pair

_bootstrap_builtin_tasks()
task_spec = get_task(TASK)

discovery_cfg = {
    "algorithm": "eap-ig", "task": TASK, "level": "node",
    "batch_size": BATCH_SIZE,
    "data_params": {"num_examples": N_EXAMPLES, "batch_size": BATCH_SIZE},
}
device = str(model.cfg.device)
dataloader = task_spec.build_dataloader(model, discovery_cfg, device)
metric_fn = task_spec.metric_fn()


def measure_bias(m, dl):
    """Mean signed logit-diff over the task (higher magnitude = stronger
    gendered-pronoun preference). Task dataloaders yield (clean, corrupted,
    label[, answer_spans]) tuples \u2014 NOT dicts \u2014 so tokenize the pair to
    get tokens + per-example prompt lengths, exactly as evaluate_graph does."""
    total, n = 0.0, 0
    with torch.no_grad():
        for batch_data in dl:
            if len(batch_data) == 4:
                clean, corrupted, label, _ = batch_data
            else:
                clean, corrupted, label = batch_data
            clean_tokens, _, clean_attn, _, input_lengths, _ = tokenize_batch_pair(
                m, clean, corrupted,
                pair_padding_side=getattr(dl, "pair_padding_side", None),
                templated=getattr(dl, "templated", False),
            )
            logits = m(clean_tokens, attention_mask=clean_attn)
            ld = metric_fn(logits, logits, input_lengths, label, loss=False, mean=True)
            total += ld.item()
            n += 1
    return total / max(n, 1)


baseline_bias = measure_bias(model, dataloader)
print(f"\nBaseline bias signal (logit-diff): {baseline_bias:.4f}")
print("Higher magnitude = stronger gendered-pronoun preference.")

## 2. Discover the bias circuit

EAP-IG attributes each node's contribution to the gendered-pronoun logit-diff. Nodes with high scores are the ones *driving* the bias.

In [ ]:
output_path = str(RESULTS_DIR / "gender_bias_circuit.pt")

circuit = ck.discover(
    model, TASK,
    algorithm="eap-ig",
    n_examples=N_EXAMPLES,
    batch_size=BATCH_SIZE,
    sparsity=0.3,
    output_path=output_path,
)

print(f"\nDiscovered bias circuit: {len(circuit.scores)} scored nodes")
print(f"\nTop 10 bias-driving nodes:")
top_nodes = sorted(circuit.scores.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
for name, score in top_nodes:
    node_type = "attn" if name.startswith("A") else "MLP"
    print(f"  {name:12s}  ({node_type})  score={abs(score):.4f}")

# Count by type
attn_count = sum(1 for n, s in top_nodes if n.startswith("A"))
mlp_count = len(top_nodes) - attn_count
print(f"\nTop-10 breakdown: {attn_count} attention heads, {mlp_count} MLP layers")
print("(Consistent with literature: bias localizes to sparse attention heads)")

## 3. Audit: Faithfulness evaluation

Confirm the discovered circuit is causally responsible for the bias, not an artifact of the attribution method. We run the fast pillars plus baselines to verify the circuit outperforms random node selection.

Backend: `circuitkit.evaluation.full.run_full_faithfulness` runs patching (Pillar 1), ablation (Pillar 2), and baseline comparisons (Pillar 5).

In [ ]:
report = ck.faithfulness(
    model, circuit, TASK,
    pillars=["patching", "ablation", "baselines"],
    n_examples=min(N_EXAMPLES, 64),
    batch_size=BATCH_SIZE,
)

print("INITIAL BIAS AUDIT")
print("=" * 50)
_ps, _as = report.patching_score, report.ablation_score  # Optional[float]
print(f"  Causal patching:  {_ps:.3f}" if _ps is not None else "  Causal patching:  invalid")
print(f"  Ablation:         {_as:.3f}" if _as is not None else "  Ablation:         invalid")
# Pillar 5 output lives in report.baseline_comparison (a dict), not a
# scalar report.baselines_score attribute (which does not exist).
if isinstance(report.baseline_comparison, dict):
    print(f"  vs Baselines:     {report.baseline_comparison.get('summary', report.baseline_comparison)}")
print()
print("Interpretation:")
print("  High patching = restoring these nodes recovers bias behavior")
print("  High ablation = removing these nodes degrades bias behavior")
print("  Both high = circuit is the genuine bias mechanism")

## 4. Mitigate: Selective fine-tuning targeting

Use the circuit scores to identify the components most responsible for bias. `ck.selective_finetune()` returns a `SelectionResult` that maps exactly which attention heads and MLP layers should be targeted for bias-reduction training.

Backend: `circuitkit.applications.selective_finetuning.selector.select_components` ranks nodes by circuit score and selects the top fraction.

In [ ]:
selection = ck.selective_finetune(
    circuit,
    model_name=MODEL_NAME,
    top_fraction=0.15,
    scope="both",
)

print("Selective fine-tuning targets (top 15% bias-critical components):")
print(f"  Attention heads: {len(selection.attn)} selected")
for key, indices in list(selection.attn.items())[:5]:
    print(f"    {key}: {indices}")
print(f"  MLP layers: {len(selection.mlp)} selected")
for key, indices in list(selection.mlp.items())[:5]:
    print(f"    {key}: {indices}")
print()
print("These components can be targeted with LoRA / gradient masking")
print("to reduce bias while leaving the rest of the model frozen.")

In [ ]:
# Apply structural pruning at the bias-circuit nodes as a direct mitigation
# (pruning the bias mechanism is the fastest path to bias reduction)
pruned_model = ck.prune(
    model, circuit,
    sparsity=0.2,
    scope="both",
)

print("Applied structural pruning at 20% sparsity on bias-circuit nodes.")
print("The lowest-scoring (most bias-contributing) nodes are masked.")

## 5. Re-audit: Verify bias reduction + capability preservation

The crucial second half: confirm the fix worked *and* didn't break the model. We re-measure the bias signal on the pruned model and compare to baseline.

In [ ]:
# Re-measure bias signal on the pruned model (reuses measure_bias from above)
dataloader_reeval = task_spec.build_dataloader(pruned_model, discovery_cfg, device)
post_bias = measure_bias(pruned_model, dataloader_reeval)
bias_reduction = 1.0 - abs(post_bias) / max(abs(baseline_bias), 1e-8)

print(f"Bias signal before mitigation: {baseline_bias:.4f}")
print(f"Bias signal after mitigation:  {post_bias:.4f}")
print(f"Bias reduction:                {bias_reduction:.1%}")

In [ ]:
# Capability preservation: run a quick general-knowledge check
# The pruning targeted bias-specific nodes; general capability should survive.
general_prompts = [
    "The largest planet in our solar system is",
    "Water freezes at zero degrees",
    "The speed of light is approximately",
]

print("\nCapability preservation check (general knowledge):")
print("-" * 60)
for prompt in general_prompts:
    tokens = pruned_model.to_tokens(prompt, prepend_bos=True).to(device)
    with torch.no_grad():
        logits = pruned_model(tokens)
    next_token = pruned_model.to_string(logits[0, -1, :].argmax().unsqueeze(0))
    print(f"  {prompt}... -> {next_token.strip()}")

## 6. Audit report (compliance artifact)

In [ ]:
import json

audit_report = {
    "model": MODEL_NAME,
    "task": TASK,
    "algorithm": "eap-ig",
    "circuit_size": len(circuit.scores),
    "pre_mitigation": {
        "bias_signal": baseline_bias,
        "faithfulness_patching": report.patching_score,
        "faithfulness_ablation": report.ablation_score,
    },
    "mitigation": {
        "method": "structural_pruning",
        "sparsity": 0.2,
        "scope": "both",
        "selective_finetune_targets": {
            "attn_components": len(selection.attn),
            "mlp_components": len(selection.mlp),
        },
    },
    "post_mitigation": {
        "bias_signal": post_bias,
        "bias_reduction_pct": round(bias_reduction * 100, 1),
    },
    "verdict": "PASS" if bias_reduction > 0.1 else "REVIEW",
}

report_path = RESULTS_DIR / "bias_audit_report.json"
with open(report_path, "w") as f:
    json.dump(audit_report, f, indent=2, default=str)

print("=" * 60)
print("GENDER BIAS AUDIT REPORT")
print("=" * 60)
print(f"  Model:              {MODEL_NAME}")
print(f"  Circuit nodes:      {len(circuit.scores)}")
print(f"  Bias (before):      {baseline_bias:.4f}")
print(f"  Bias (after):       {post_bias:.4f}")
print(f"  Reduction:          {bias_reduction:.1%}")
print(f"  Verdict:            {audit_report['verdict']}")
print(f"  Report saved to:    {report_path}")
print("=" * 60)

## 7. Visualization: Before vs after

Visualize the circuit to see which components were identified as bias-critical.

In [ ]:
# Interactive circuit graph showing the bias mechanism
viz = ck.visualize_circuit(circuit, mode="graph", output=str(RESULTS_DIR / "bias_circuit.html"))
print(f"Circuit visualization saved to {RESULTS_DIR / 'bias_circuit.html'}")

# In Jupyter: display inline
try:
    circuit.plot()
except Exception:
    print("(Inline plot requires Jupyter with ipywidgets)")

## Interpretation guide

| Metric | What it means |
|--------|---------------|
| **Bias signal (logit-diff)** | How strongly the model prefers one gendered pronoun over another. Closer to 0 = less biased. |
| **Faithfulness scores** | Confirm the circuit is the *cause* of the bias, not a correlation. High patching + ablation = genuine mechanism. |
| **Bias reduction %** | Relative decrease in bias signal after mitigation. >10% is meaningful; >30% is strong. |
| **Capability preservation** | General-knowledge completions should remain coherent. Bias mitigation should not degrade unrelated tasks. |

### The full responsible-AI loop

1. **Detect:** Discovery finds the bias circuit (which heads/MLPs drive gendered predictions)
2. **Validate:** Faithfulness evaluation confirms the circuit is causal, not spurious
3. **Fix:** Selective fine-tuning / structural pruning targets only the bias components
4. **Prove:** Re-measurement shows bias down + general capability preserved
5. **Document:** Audit report is a structured JSON artifact for compliance teams

### Key API calls used

| API | Backend module | Purpose |
|-----|---------------|----------|
| `ck.discover()` | `circuitkit.backends.eap.attribute_node` | Identify bias circuit |
| `ck.faithfulness()` | `circuitkit.evaluation.full` | Validate circuit causality |
| `ck.selective_finetune()` | `circuitkit.applications.selective_finetuning.selector` | Identify bias-critical components for LoRA/masking |
| `ck.prune()` | `circuitkit.applications.pruning.StructuralPruner` | Directly mask bias nodes |
| `ck.visualize()` | `circuitkit.visualize.graph_viz` | Interactive circuit graph |